# 📘 Semaine 4 — EXTI + NVIC

**Cours :** Microcontrôleurs STM32F103C6T6  
**Durée :** 4h30 (1h30 cours + 1h30 atelier + 1h30 homework)  
**Enseignant :** ____________________  
**Étudiant :** ____________________  
**Date :** ____________________

---

## 🎯 Objectifs pédagogiques de la semaine

À la fin de cette semaine, l'étudiant sera capable de :

1. **Expliquer** le mécanisme d'interruption matérielle et le rôle du NVIC.
2. **Configurer** une ligne EXTI sur un GPIO (front montant, descendant ou les deux).
3. **Écrire** une ISR et la connecter à EXTI via HAL.
4. **Gérer** les priorités d'interruption (préemption et sous-priorité).
5. **Diagnostiquer** un problème d'interruption (rebonds, IRQ non déclenchée, priorité bloquante).

---

## 🗺️ Plan de la semaine

| Partie | Contenu | Durée |
|---|---|---|
| **A — Cours** | Activités 1 à 5 | 1h30 |
| **B — Atelier** | TP4 : Compteur d'impulsions par interruption | 1h30 |
| **C — Homework** | Exercices 1 à 3 | 1h30 |
| **D — Auto-évaluation** | Checklist finale | 5 min |

---
# 🎓 PARTIE A — COURS INTÉGRÉ (1h30)

## 🔹 Activité 1 — Rappel & mise en contexte (10 min)

### 🔄 Rappel des semaines précédentes
- **Cortex-M3** @ 72 MHz — pipeline 3 étages.
- **GPIO** : registres CRL / CRH / IDR / ODR / BSRR.
- **NVIC** : 16 niveaux de priorité, jusqu'à 240 IRQ (43 sur le STM32F103C6T6).
- **APB2** : 72 MHz — GPIO, EXTI, AFIO.

### ✍️ Questions flash (2 min)
1. Que veut dire NVIC ? → ...
2. Combien de broches par port GPIO ? → ...
3. Quelle macro active l'horloge d'AFIO ? → ...
4. Que signifie ISR ? → ...

---

## 🔹 Activité 2 — Principe des interruptions (20 min)

### 📖 2.1 — Polling vs Interruption

| Critère | Polling | Interruption |
|---|---|---|
| Principe | Le CPU teste périodiquement | Le périphérique signale l'événement |
| Consommation | Élevée (boucle active) | Faible |
| Latence | Dépend de la période | Faible et déterministe |
| Complexité | Simple | Nécessite NVIC + ISR |
| Perte d'événements | Possible si trop rapide | Non (si priorité adaptée) |

> 💡 **Interruption** = signal matériel qui **suspend** le programme principal et **exécute** une routine dédiée (ISR).

### 📖 2.2 — Chaîne d'une interruption

```
Événement externe
       │
       ▼
  ┌──────────┐
  │   EXTI   │  (détection front/level)
  └────┬─────┘
       │ IRQ
       ▼
  ┌──────────┐
  │   NVIC   │  (priorité, masquage, pending)
  └────┬─────┘
       │
       ▼
  ┌──────────────┐
  │  Cortex-M3   │  (sauvegarde contexte, branche ISR)
  └────┬─────────┘
       │
       ▼
  ┌──────────────┐
  │     ISR      │  (exécution du code utilisateur)
  └────┬─────────┘
       │
       ▼
  Retour programme principal
```

### 📖 2.3 — Vocabulaire

| Terme | Définition |
|---|---|
| **IRQ** | *Interrupt Request* — demande d'interruption |
| **ISR** | *Interrupt Service Routine* — fonction appelée par l'IRQ |
| **Vecteur** | Adresse de l'ISR, stockée dans la *vector table* |
| **NVIC** | *Nested Vectored Interrupt Controller* |
| **Pending** | IRQ en attente (levée mais pas encore traitée) |
| **Préemption** | Une IRQ prioritaire peut interrompre une ISR en cours |

---

## 🔹 Activité 3 — Le NVIC (20 min)

### 📖 3.1 — Organisation

Le NVIC est **intégré au cœur Cortex-M3**. Il gère :
- **240 IRQ** max (43 sur STM32F103C6T6)
- **16 niveaux de priorité** (4 bits utiles)
- **Préemption** (nested)
- **Pendings** (mise en file d'attente)

### 📖 3.2 — Registres NVIC (extrait)

| Registre | Rôle |
|---|---|
| `ISER[n]` | *Interrupt Set-Enable Register* — activer une IRQ |
| `ICER[n]` | *Interrupt Clear-Enable Register* — désactiver |
| `ISPR[n]` | *Set-Pending* — forcer une IRQ en attente |
| `ICPR[n]` | *Clear-Pending* — effacer un pending |
| `IABR[n]` | *Active Bit Register* — lire les IRQ actives |
| `IPR[n]` | *Interrupt Priority Register* — priorité par IRQ |

### 📖 3.3 — Priorités et préemption

Le STM32F103 utilise **4 bits** de priorité, configurables en **groupes** :

| Groupe (PRIGROUP) | Bits préemption | Bits sous-priorité | Niveaux |
|---|---|---|---|
| 0 | 4 | 0 | 16 préemptions |
| 1 | 3 | 1 | 8 × 2 |
| 2 | 2 | 2 | 4 × 4 |
| 3 | 1 | 3 | 2 × 8 |
| 4 | 0 | 4 | Aucune préemption |

**Règle :** plus la valeur numérique est **petite**, plus la priorité est **élevée**.

```
Priorité 0  ──▶ la plus haute
Priorité 15 ──▶ la plus basse
```

### 📖 3.4 — Préemption vs sous-priorité

```
Scénario : IRQ_A priorité (2,0), IRQ_B priorité (1,1)

IRQ_B arrive ──▶ préempte IRQ_A (car préemption 1 < 2)
IRQ_A arrive ──▶ NE préempte PAS IRQ_B (même préemption 1)
                 → traitement séquentiel après IRQ_B
```

> ⚠️ Une ISR ne peut être préemptée que par une IRQ de **préemption strictement plus élevée** (numériquement plus petite).

### 🐍 Simulation Python — Priorités NVIC (10 min)

Simulons l'arrivée d'IRQ avec différentes priorités pour visualiser la préemption.

In [ ]:
# ============================================================
# Simulateur de priorités NVIC (préemption + sous-priorité)
# ============================================================

class NVIC:
    def __init__(self):
        self.priorites = {}     # nom -> (preemption, sub)
        self.actives   = []     # pile des IRQ en cours

    def set_priorite(self, irq, preemption, sub):
        self.priorites[irq] = (preemption, sub)

    def arriver(self, irq):
        prio = self.priorites.get(irq, (15, 0))
        if not self.actives:
            self.actives.append((irq, prio))
            return f"▶ {irq} démarre (priorité {prio})"
        # Compare avec l'IRQ en cours
        courante = self.actives[-1]
        if prio[0] < courante[1][0]:
            self.actives.append((irq, prio))
            return f"⤴ {irq} PRÉEMPTE {courante[0]} (prio {prio} < {courante[1]})"
        else:
            return f"⏸ {irq} en attente (prio {prio} ≥ {courante[1]})"

    def terminer(self):
        if not self.actives:
            return "(aucune IRQ active)"
        irq, prio = self.actives.pop()
        return f"⏹ {irq} terminée"

    def etat(self):
        if not self.actives:
            return "[CPU: programme principal]"
        return "[Pile: " + " → ".join(i for i, _ in self.actives) + "]"

# --- Scénario ---
nvic = NVIC()
nvic.set_priorite("EXTI0",  preemption=2, sub=0)
nvic.set_priorite("TIM2",   preemption=1, sub=1)
nvic.set_priorite("USART1", preemption=1, sub=0)

print("📋 Priorités configurées :")
for irq, p in nvic.priorites.items():
    print(f"   {irq:<8} → préemption={p[0]}, sous-prio={p[1]}")
print()

for action in ["arriver EXTI0", "arriver TIM2", "arriver USART1", "terminer", "terminer", "terminer"]:
    if action.startswith("arriver"):
        irq = action.split()[1]
        print(f"{nvic.arriver(irq):<55} {nvic.etat()}")
    else:
        print(f"{nvic.terminer():<55} {nvic.etat()}")

---

## 🔹 Activité 4 — Le contrôleur EXTI (25 min)

### 📖 4.1 — Vue d'ensemble

Le **EXTI** (*External Interrupt/Event Controller*) détecte les **fronts** (montant, descendant) ou les **niveaux** sur une ligne et génère une IRQ vers le NVIC.

Le STM32F103C6T6 possède **16 lignes EXTI** (EXTI0 à EXTI15) — plus les lignes PVD, RTC, USB, Ethernet (non exposées).

### 📖 4.2 — Mapping GPIO ↔ EXTI

Chaque ligne EXTI est connectée à **une seule broche** parmi tous les ports.

| Ligne EXTI | Broches possibles |
|---|---|
| EXTI0 | PA0, PB0, PC0, … |
| EXTI1 | PA1, PB1, PC1, … |
| … | … |
| EXTI15 | PA15, PB15, PC15, … |

**Registre de sélection :** `AFIO_EXTICR1..4` — chaque registre gère 4 lignes.

```
AFIO_EXTICR1 (EXTI0..3) :  4 bits par ligne
  0b0000 → PAx
  0b0001 → PBx
  0b0010 → PCx
  0b0011 → PDx
  ...
```

> ⚠️ **Contrainte** : une seule broche par ligne EXTI à un instant donné.  
> On ne peut **pas** avoir EXTI0 sur PA0 **et** PB0 simultanément.

### 📖 4.3 — Registres EXTI

| Registre | Rôle |
|---|---|
| `IMR` | *Interrupt Mask Register* — masque (1 = activée) |
| `EMR` | *Event Mask Register* — masque pour événements |
| `RTSR` | *Rising Trigger Selection* — front montant |
| `FTSR` | *Falling Trigger Selection* — front descendant |
| `SWIER` | *Software Interrupt Event Register* — forcer par logiciel |
| `PR` | *Pending Register* — drapeau d'IT (à acquitter) |

### 📖 4.4 — Chaîne EXTI complète

```
   PA0 ──┐
   PB0 ──┼──▶ AFIO_EXTICR1 ──▶ EXTI0 ──┐
   PC0 ──┘                              │
                                        ▼
                                  ┌──────────────┐
                                  │  Edge Detect │
                                  │  (RTSR/FTSR) │
                                  └──────┬───────┘
                                         │
                                    ┌────▼────┐
                                    │   IMR   │ (masque)
                                    └────┬────┘
                                         │
                                    ┌────▼────┐
                                    │   PR    │ (pending)
                                    └────┬────┘
                                         │
                                         ▼
                                        NVIC
```

### 📖 4.5 — Vecteurs d'interruption

Chaque ligne EXTI a son propre vecteur dans le NVIC :

| Ligne | IRQn |
|---|---|
| EXTI0 | `EXTI0_IRQn` |
| EXTI1 | `EXTI1_IRQn` |
| EXTI2 | `EXTI2_IRQn` |
| EXTI3 | `EXTI3_IRQn` |
| EXTI4 | `EXTI4_IRQn` |
| EXTI5–9 | `EXTI9_5_IRQn` (vecteur commun) |
| EXTI10–15 | `EXTI15_10_IRQn` (vecteur commun) |

> 💡 Les lignes 5 à 9 et 10 à 15 partagent un même vecteur → il faut identifier la source dans l'ISR via `PR`.

### 🐍 Simulation Python — Détection de fronts (10 min)

Simulons un signal sur une ligne EXTI et comptons les fronts.

In [ ]:
# ============================================================
# Détection de fronts sur une ligne EXTI
# ============================================================

def detecter_fronts(signal, rtsr=True, ftsr=False):
    """
    Retourne la liste des indices de fronts détectés.
    signal : liste de 0/1 (états échantillonnés)
    rtsr   : front montant activé
    ftsr   : front descendant activé
    """
    fronts = []
    for i in range(1, len(signal)):
        if rtsr and signal[i-1] == 0 and signal[i] == 1:
            fronts.append((i, "↑ montant"))
        if ftsr and signal[i-1] == 1 and signal[i] == 0:
            fronts.append((i, "↓ descendant"))
    return fronts

# Signal : 0 0 1 1 0 1 0 1 1 0 0 1
signal = [0,0,1,1,0,1,0,1,1,0,0,1]
print("Signal EXTI0 :", " ".join(map(str, signal)))
print()

print("RTSR=1, FTSR=0 (front montant seulement) :")
for i, t in detecter_fronts(signal, rtsr=True, ftsr=False):
    print(f"   t={i} : {t}")

print("\nRTSR=0, FTSR=1 (front descendant seulement) :")
for i, t in detecter_fronts(signal, rtsr=False, ftsr=True):
    print(f"   t={i} : {t}")

print("\nRTSR=1, FTSR=1 (les deux) :")
for i, t in detecter_fronts(signal, rtsr=True, ftsr=True):
    print(f"   t={i} : {t}")

---

## 🔹 Activité 5 — Programmation EXTI avec HAL (15 min)

### 📖 5.1 — Configuration CubeMX

1. Cliquer sur **PA0** → **GPIO_EXTI0**
2. Onglet **GPIO** → configuration de la broche :
   - **GPIO mode** : `External Interrupt Mode with Rising edge trigger detection`
   - **Pull-up/Pull-down** : selon le câblage (ex. `Pull-down`)
3. Onglet **NVIC** → cocher `EXTI line0 interrupt`
4. Régler la priorité (Preemption Priority / Sub Priority)
5. Générer le code

### 📖 5.2 — Code généré automatiquement

```c
/* Dans MX_GPIO_Init() */
GPIO_InitStruct.Pin  = GPIO_PIN_0;
GPIO_InitStruct.Mode = GPIO_MODE_IT_RISING;
GPIO_InitStruct.Pull = GPIO_PULLDOWN;
HAL_GPIO_Init(GPIOA, &GPIO_InitStruct);

/* Dans MX_NVIC_Init() (ou équivalent) */
HAL_NVIC_SetPriority(EXTI0_IRQn, 1, 0);
HAL_NVIC_EnableIRQ(EXTI0_IRQn);
```

### 📖 5.3 — Écrire l'ISR (callback HAL)

HAL utilise un **callback** : pas besoin de définir l'ISR directement.

In [ ]:
/* ============================================================
   ISR EXTI0 par callback HAL
   ============================================================ */

/* --- Callback utilisateur (appelé par HAL_GPIO_EXTI_IRQHandler) --- */
void HAL_GPIO_EXTI_Callback(uint16_t GPIO_Pin)
{
    if (GPIO_Pin == GPIO_PIN_0)
    {
        HAL_GPIO_TogglePin(GPIOC, GPIO_PIN_13);
    }
}

/* --- Le handler est déjà défini par HAL dans stm32f1xx_it.c --- */
// void EXTI0_IRQHandler(void)
// {
//     HAL_GPIO_EXTI_IRQHandler(GPIO_PIN_0);
// }

### 📖 5.4 — Version « registres » sans HAL

Pour comprendre ce qui se passe sous le capot :

In [ ]:
/* ============================================================
   EXTI0 en accès direct registres (sans HAL)
   ============================================================ */

// 1) Activer les horloges AFIO + GPIOA (sur APB2)
RCC->APB2ENR |= RCC_APB2ENR_AFIOEN | RCC_APB2ENR_IOPAEN;

// 2) Configurer PA0 en entrée flottante (CRL)
GPIOA->CRL &= ~(0xF << 0);          // effacer PA0
GPIOA->CRL |=  (0x4 << 0);          // 0b0100 : entrée flottante

// 3) Mapper EXTI0 sur PA0 (AFIO_EXTICR1 : bits [3:0] = 0b0000)
AFIO->EXTICR[0] &= ~0x000F;
AFIO->EXTICR[0] |=  0x0000;         // 0 = PA0

// 4) Activer le front montant sur EXTI0
EXTI->RTSR |= (1 << 0);
EXTI->FTSR &= ~(1 << 0);

// 5) Dé-masquer EXTI0
EXTI->IMR |= (1 << 0);

// 6) Activer l'IRQ dans le NVIC (EXTI0 = IRQ 6)
NVIC_EnableIRQ(EXTI0_IRQn);
NVIC_SetPriority(EXTI0_IRQn, 1);

/* ISR utilisateur */
void EXTI0_IRQHandler(void)
{
    if (EXTI->PR & (1 << 0))         // vérifier que c'est bien EXTI0
    {
        EXTI->PR = (1 << 0);         // acquitter (écrire 1)
        GPIOC->ODR ^= (1 << 13);     // toggle PC13
    }
}

### 🔍 Comparaison HAL vs registres

| Étape | HAL | Registres |
|---|---|---|
| Activation horloge | `__HAL_RCC_GPIOA_CLK_ENABLE()` | `RCC->APB2ENR \|= ...` |
| Config broche | `HAL_GPIO_Init()` | `GPIOA->CRL` |
| Mapping EXTI | via CubeMX | `AFIO->EXTICR[]` |
| Activation IRQ | `HAL_NVIC_EnableIRQ()` | `NVIC_EnableIRQ()` |
| Priorité | `HAL_NVIC_SetPriority()` | `NVIC_SetPriority()` |
| ISR | callback HAL | handler CMSIS direct |
| Acquittement | automatique | `EXTI->PR = ...` manuel |

---

## 🔹 Activité 6 — QCM formatif (10 min)

**1. Combien de lignes EXTI disponibles sur le STM32F103C6T6 ?**  
A. 4  
B. 8  
C. 16  
D. 32

**2. Le registre qui sélectionne le port pour EXTI0 est :**  
A. `EXTI->IMR`  
B. `AFIO->EXTICR[0]`  
C. `NVIC->ISER[0]`  
D. `GPIOA->CRL`

**3. Une ISR en cours peut être préemptée si la nouvelle IRQ a :**  
A. La même priorité  
B. Une priorité de préemption plus élevée (numériquement plus petite)  
C. Une priorité de préemption plus basse  
D. Une sous-priorité plus élevée

**4. Le registre `EXTI->PR` sert à :**  
A. Activer EXTI  
B. Acquitter un pending  
C. Lire l'état des pins  
D. Configurer le front

**5. Sur le STM32F103, EXTI15 et EXTI10 partagent :**  
A. Le même registre IMR  
B. Le même vecteur NVIC  
C. La même broche  
D. Le même RTSR

**6. Quelle est la priorité la plus haute dans le NVIC ?**  
A. 0  
B. 1  
C. 15  
D. 255

### ✅ Corrigé du QCM formatif

| Q | Réponse | Explication |
|---|---|---|
| 1 | **C — 16** | EXTI0 à EXTI15 |
| 2 | **B — `AFIO->EXTICR[0]`** | Mapping GPIO ↔ EXTI |
| 3 | **B — Préemption plus élevée** | Numériquement plus petite |
| 4 | **B — Acquitter un pending** | Écrire 1 pour effacer |
| 5 | **B — Même vecteur NVIC** | `EXTI15_10_IRQn` |
| 6 | **A — 0** | Plus petit = plus prioritaire |

**Mon score : ___ / 6**

---

# 🛠️ PARTIE B — ATELIER / TP (1h30)

## 🧪 TP4 — Compteur d'impulsions par interruption

### 🎯 Objectif
Compter les appuis sur un bouton via EXTI, afficher le compteur sur 8 LED, et comparer polling vs interruption.

### 📋 Tâches à réaliser (par binôme)

| # | Tâche | Durée | Livrable |
|---|---|---|---|
| 1 | Configurer EXTI0 sur PA0 (front montant, pull-down) | 15 min | Capture CubeMX |
| 2 | Écrire `HAL_GPIO_EXTI_Callback` pour incrémenter un compteur | 15 min | Code |
| 3 | Afficher le compteur sur 8 LED (binaire) | 15 min | Code + démo |
| 4 | Ajouter un bouton reset sur PA1 (EXTI1, front montant) | 15 min | Code |
| 5 | Mesurer la latence avec un oscilloscope | 15 min | Mesure |
| 6 | Comparer polling vs interruption | 10 min | Analyse |
| 7 | Rédiger le compte-rendu | 5 min | CR |

### ⚙️ Code complet — Compteur + reset + affichage 8 LED

In [ ]:
/* ============================================================
   TP4 - Compteur d'impulsions par EXTI
   - Bouton principal : PA0  (EXTI0, front montant)
   - Bouton reset     : PA1  (EXTI1, front montant)
   - LEDs             : PB0..PB7 (8 LEDs, binaire)
   ============================================================ */

#include "main.h"

volatile uint8_t compteur = 0;
volatile uint8_t reset_demande = 0;

/* --- Affichage binaire sur PB0..PB7 --- */
static void afficher_compteur(uint8_t val)
{
    // Écriture atomique de l'octet bas du port B (bits 0..7)
    GPIOB->ODR = (GPIOB->ODR & 0xFF00) | val;
}

/* --- Callback HAL EXTI --- */
void HAL_GPIO_EXTI_Callback(uint16_t GPIO_Pin)
{
    if (GPIO_Pin == GPIO_PIN_0)
    {
        compteur++;
        afficher_compteur(compteur);
    }
    else if (GPIO_Pin == GPIO_PIN_1)
    {
        reset_demande = 1;
    }
}

int main(void)
{
    HAL_Init();
    SystemClock_Config();
    MX_GPIO_Init();

    afficher_compteur(compteur);

    while (1)
    {
        if (reset_demande)
        {
            reset_demande = 0;
            compteur = 0;
            afficher_compteur(compteur);
        }
    }
}

### 🔍 Analyse du code

| Élément | Rôle |
|---|---|
| `volatile` | Empêche le compilateur d'optimiser la variable (modifiée par ISR) |
| Callback | Appelé automatiquement par `HAL_GPIO_EXTI_IRQHandler` |
| Écriture ODR partielle | Évite de modifier les bits hauts (PB8–PB15) |
| Reset différé | Traitement hors ISR (principe : ISR courte) |

### 📖 Bonnes pratiques dans une ISR

✅ **À faire :**
- ISR **courte** (< quelques µs idéalement)
- Variables partagées en `volatile`
- Poser un **flag** pour traitement hors ISR si travail long
- Acquitter le pending (fait par HAL)

❌ **À éviter :**
- `HAL_Delay()` dans une ISR (bloque le CPU)
- `printf()` dans une ISR (lent, peut causer deadlock)
- Boucles longues
- Accès non atomiques aux variables 32 bits (utiliser sections critiques)

```c
/* Section critique : désactiver temporairement les IRQ */
uint32_t primask = __get_PRIMASK();
__disable_irq();
// ... accès atomique à une variable partagée ...
__set_PRIMASK(primask);
```

### 🐍 Simulation Python — Polling vs Interruption (15 min)

Simulons les deux approches pour comparer la latence et l'efficacité.

In [ ]:
# ============================================================
# Polling vs Interruption — simulation pédagogique
# ============================================================

import random
random.seed(0)

# Scénario : bouton appuyé pendant 30 ms, on échantillonne à 1 ms
DUREE_MS       = 200
APPUI_DEBUT    = 50
APPUI_FIN      = 80
PERIODE_POLL   = 10    # ms entre deux lectures en polling

signal = [1 if APPUI_DEBUT <= t < APPUI_FIN else 0 for t in range(DUREE_MS)]

# --- Polling ---
evenements_polling = []
for t in range(0, DUREE_MS, PERIODE_POLL):
    if signal[t] == 1 and (t == 0 or signal[t - 1] == 0):
        evenements_polling.append(t)

# --- Interruption (front montant) ---
evenements_it = []
for t in range(1, DUREE_MS):
    if signal[t-1] == 0 and signal[t] == 1:
        evenements_it.append(t)

print("📊 Comparaison polling vs interruption")
print("-" * 55)
print(f"Durée simulée        : {DUREE_MS} ms")
print(f"Appui réel            : t={APPUI_DEBUT} → {APPUI_FIN} ms")
print(f"Période polling       : {PERIODE_POLL} ms")
print()
print(f"Détections polling    : {len(evenements_polling)}  (t={evenements_polling})")
print(f"Latence polling       : jusqu'à {PERIODE_POLL} ms")
print()
print(f"Détections IT         : {len(evenements_it)}  (t={evenements_it})")
print(f"Latence IT            : < 1 ms (négligeable)")
print()
print("📌 Conclusion :")
print("   → Polling : latence fixe, CPU occupé, peut rater des événements courts.")
print("   → Interruption : latence quasi nulle, CPU libre entre événements.")

### 🐍 Calcul du temps CPU consommé (variante)

In [ ]:
# Estimation CPU : polling vs interruption
FREQ_MHZ = 72
COUT_LECTURE_CYCLES = 10    # lecture GPIO + test en boucle serrée

def cpu_polling(duree_s, periode_poll_us):
    nb_lectures = (duree_s * 1e6) / periode_poll_us
    return nb_lectures * COUT_LECTURE_CYCLES

def cpu_it(nb_evenements, cout_isr_cycles=200):
    return nb_evenements * cout_isr_cycles

duree_s = 1.0

print("⚙️  Consommation CPU sur 1 seconde (72 MHz = 72 000 000 cycles)\n")
print(f"{'Approche':<28}{'Cycles':<15}{'% CPU'}")
print("-" * 55)

for periode_us in [1000, 100, 10, 1]:
    c = cpu_polling(duree_s, periode_us)
    pct = c / (FREQ_MHZ * 1e6) * 100
    print(f"Polling @ {periode_us:>5} µs         {int(c):<15}{pct:.4f} %")

# Interruption : 10 événements par seconde
for nb_evt in [1, 10, 100, 1000]:
    c = cpu_it(nb_evt)
    pct = c / (FREQ_MHZ * 1e6) * 100
    print(f"IT avec {nb_evt:>5} événements     {int(c):<15}{pct:.4f} %")

### 📝 Compte-rendu de TP4

**Nom :** __________________  **Prénom :** __________________  **Binôme :** __________________

**1. Configuration CubeMX**
- PA0 : mode ... , front ... , pull ...
- PA1 : mode ... , front ... , pull ...
- PB0–PB7 : mode ...
- Priorité EXTI0 : préemption = ... , sub = ...
- Priorité EXTI1 : préemption = ... , sub = ...

**2. Code ajouté dans `main.c`**
```c
// Colle ici ton code
```

**3. Observation**
- Le compteur s'incrémente-t-il correctement à chaque appui ? ...
- Le reset fonctionne-t-il ? ...
- Y a-t-il des comptages parasites (rebonds) ? ...

**4. Mesure de latence**
- Latence mesurée sur oscilloscope : ... µs
- Comparaison avec la latence théorique du Cortex-M3 (~12 cycles) : ...

**5. Comparaison polling vs interruption**
- CPU occupé en polling : ... %
- CPU occupé en IT : ... %
- Conclusion : ...

**6. Problèmes rencontrés**
- ...

**7. Solutions apportées**
- ...

### 🧪 Exercice bonus — Deux boutons, deux priorités

Ajouter un second bouton sur **PA2 (EXTI2)** avec une priorité **plus élevée** que EXTI0.

**Cahier des charges :**
- EXTI2 (PA2) : priorité préemption = 0
- EXTI0 (PA0) : priorité préemption = 1
- EXTI0 fait `HAL_Delay(100)` avant d'incrémenter (volontairement long)
- EXTI2 incrémente un compteur séparé immédiatement
- Observer à l'oscilloscope : EXTI2 préempte-t-il EXTI0 ?

**Questions :**
1. Le compteur EXTI0 est-il mis à jour correctement ?
2. Le compteur EXTI2 est-il mis à jour pendant le `HAL_Delay` d'EXTI0 ?
3. Que se passerait-il si les deux avaient la même préemption ?

In [ ]:
// Squelette solution bonus

volatile uint16_t cnt0 = 0;
volatile uint16_t cnt2 = 0;

void HAL_GPIO_EXTI_Callback(uint16_t GPIO_Pin)
{
    if (GPIO_Pin == GPIO_PIN_0)   // priorité 1
    {
        HAL_Delay(100);           // simule traitement long
        cnt0++;
    }
    else if (GPIO_Pin == GPIO_PIN_2)  // priorité 0
    {
        cnt2++;
    }
}

---

# 🏠 PARTIE C — HOMEWORK (1h30)

## 📚 Exercices à rendre

### 🧩 Exercice 1 — Mapping EXTI (30 min)

Compléter le tableau suivant en indiquant les **bits à écrire** dans `AFIO_EXTICR1..4` pour chaque configuration.

| Ligne EXTI | Broche | Registre EXTICR | Valeur 4 bits |
|---|---|---|---|
| EXTI0 | PA0 | EXTICR[0] bits [3:0] | ? |
| EXTI1 | PB1 | EXTICR[0] bits [7:4] | ? |
| EXTI2 | PC2 | EXTICR[0] bits [11:8] | ? |
| EXTI3 | PA3 | EXTICR[0] bits [15:12] | ? |
| EXTI4 | PB4 | EXTICR[1] bits [3:0] | ? |
| EXTI5 | PA5 | EXTICR[1] bits [7:4] | ? |
| EXTI6 | PC6 | EXTICR[1] bits [11:8] | ? |
| EXTI7 | PA7 | EXTICR[1] bits [15:12] | ? |
| EXTI8 | PB8 | EXTICR[2] bits [3:0] | ? |
| EXTI9 | PA9 | EXTICR[2] bits [7:4] | ? |
| EXTI10 | PC10 | EXTICR[2] bits [11:8] | ? |
| EXTI11 | PA11 | EXTICR[2] bits [15:12] | ? |
| EXTI12 | PB12 | EXTICR[3] bits [3:0] | ? |
| EXTI13 | PC13 | EXTICR[3] bits [7:4] | ? |

**Codes de port :** `0000` = A, `0001` = B, `0010` = C

In [ ]:
# Corrigé Exercice 1

PORTS = {'A': 0b0000, 'B': 0b0001, 'C': 0b0010}

configs = [
    (0,  'A'), (1,  'B'), (2,  'C'), (3,  'A'),
    (4,  'B'), (5,  'A'), (6,  'C'), (7,  'A'),
    (8,  'B'), (9,  'A'), (10, 'C'), (11, 'A'),
    (12, 'B'), (13, 'C'),
]

print(f"{'Ligne':<8}{'Broche':<10}{'EXTICR':<10}{'Bits':<10}{'Valeur'}")
print("-" * 50)
for ligne, port in configs:
    reg_idx = ligne // 4
    shift   = (ligne % 4) * 4
    val     = PORTS[port]
    print(f"EXTI{ligne:<4}P{port}{ligne:<8}EXTICR[{reg_idx}]  [{shift+3}:{shift}]  0b{val:04b}")

### 🧩 Exercice 2 — Priorités et préemption (30 min)

Soit 3 IRQ :
- **IRQ_A** : préemption = 2, sub = 0
- **IRQ_B** : préemption = 1, sub = 3
- **IRQ_C** : préemption = 1, sub = 1

**Questions :**

1. Quelle IRQ a la priorité la plus élevée ? La plus basse ?
2. Si IRQ_A est en cours d'exécution, puis IRQ_B arrive : est-elle préemptée ?
3. Si IRQ_B est en cours d'exécution, puis IRQ_C arrive : est-elle préemptée ?
4. Si IRQ_B est en cours d'exécution, puis IRQ_A arrive : est-elle préemptée ?
5. Ordre d'exécution complet si les 3 arrivent dans l'ordre A → B → C ?

👉 Utiliser la simulation Python `NVIC` ci-dessus pour vérifier tes réponses.

In [ ]:
# Corrigé Exercice 2 — utilise la classe NVIC définie plus haut

nvic2 = NVIC()
nvic2.set_priorite("IRQ_A", preemption=2, sub=0)
nvic2.set_priorite("IRQ_B", preemption=1, sub=3)
nvic2.set_priorite("IRQ_C", preemption=1, sub=1)

print("=== Scénario A → B → C ===")
print(nvic2.arriver("IRQ_A"))
print(nvic2.arriver("IRQ_B"))
print(nvic2.arriver("IRQ_C"))
print(nvic2.terminer())
print(nvic2.terminer())
print(nvic2.terminer())

### 🧩 Exercice 3 — Lecture du RM0008 (30 min)

Lire le **chapitre 10 (EXTI)** du RM0008 et le chapitre **4.2 (NVIC)** du PM0056.

Répondre :

1. Combien de lignes EXTI sont disponibles sur le STM32F103 ?
2. Que se passe-t-il si on écrit `1` dans `EXTI->SWIER` ?
3. Quelle est la différence entre `EXTI->IMR` et `EXTI->EMR` ?
4. Comment acquitte-t-on un pending sur EXTI ?
5. Pourquoi les lignes 5 à 9 partagent-elles un même vecteur NVIC ?
6. Que se passe-t-il si une IRQ est levée alors que son bit `IMR` est à 0 ?

### ✍️ Réponses — Exercice 3

1. ...
2. ...
3. ...
4. ...
5. ...
6. ...

---

## 🧮 Exercice supplémentaire — Simulateur EXTI complet (optionnel)

Complète la classe `EXTISim` pour modéliser :
- `mapper(ligne, port, pin)` : configure le mapping AFIO
- `config_front(ligne, rising, falling)` : RTSR / FTSR
- `activer(ligne)` / `desactiver(ligne)` : IMR
- `forcer(ligne)` : SWIER
- `injecter_signal(ligne, valeur)` : simule un changement d'état et génère un événement
- `pending(ligne)` : renvoie True/False

In [ ]:
# À compléter
class EXTISim:
    def __init__(self):
        self.mapping    = {}     # ligne -> (port, pin)
        self.rtsr       = 0
        self.ftsr       = 0
        self.imr        = 0
        self.pr         = 0      # pending
        self.etats      = {}     # (port, pin) -> dernier niveau

    def mapper(self, ligne, port, pin):
        # TODO
        pass

    def config_front(self, ligne, rising=True, falling=False):
        # TODO
        pass

    def activer(self, ligne):
        # TODO
        pass

    def desactiver(self, ligne):
        # TODO
        pass

    def forcer(self, ligne):
        # TODO
        pass

    def injecter_signal(self, ligne, niveau):
        # TODO : détecter front, lever pending si activé
        pass

    def acquitter(self, ligne):
        # TODO
        pass

    def pending(self, ligne):
        return bool(self.pr & (1 << ligne))

In [ ]:
# ✅ Corrigé
class EXTISimOK:
    def __init__(self):
        self.mapping = {}
        self.rtsr = 0
        self.ftsr = 0
        self.imr  = 0
        self.pr   = 0
        self.etats = {}

    def mapper(self, ligne, port, pin):
        self.mapping[ligne] = (port, pin)
        self.etats[(port, pin)] = 0

    def config_front(self, ligne, rising=True, falling=False):
        if rising: self.rtsr |=  (1 << ligne)
        else:      self.rtsr &= ~(1 << ligne)
        if falling: self.ftsr |=  (1 << ligne)
        else:       self.ftsr &= ~(1 << ligne)

    def activer(self, ligne):
        self.imr |= (1 << ligne)

    def desactiver(self, ligne):
        self.imr &= ~(1 << ligne)

    def forcer(self, ligne):
        self.pr |= (1 << ligne)

    def injecter_signal(self, ligne, niveau):
        port, pin = self.mapping[ligne]
        ancien = self.etats[(port, pin)]
        self.etats[(port, pin)] = niveau
        front = None
        if ancien == 0 and niveau == 1 and (self.rtsr & (1 << ligne)):
            front = "montant"
        elif ancien == 1 and niveau == 0 and (self.ftsr & (1 << ligne)):
            front = "descendant"
        if front and (self.imr & (1 << ligne)):
            self.pr |= (1 << ligne)
            return f"EXTI{ligne} : front {front} → pending"
        return f"EXTI{ligne} : niveau {niveau} (pas d'événement)"

    def acquitter(self, ligne):
        self.pr &= ~(1 << ligne)

    def pending(self, ligne):
        return bool(self.pr & (1 << ligne))

e = EXTISimOK()
e.mapper(0, 'A', 0)
e.config_front(0, rising=True, falling=False)
e.activer(0)
print(e.injecter_signal(0, 1))    # front montant → pending
print("Pending EXTI0 =", e.pending(0))
e.acquitter(0)
print("Après acquit, pending =", e.pending(0))
print(e.injecter_signal(0, 0))    # front descendant non configuré → rien

---
# ✅ PARTIE D — AUTO-ÉVALUATION Semaine 4

Coche ce que tu maîtrises.

- [ ] Je sais expliquer la différence entre polling et interruption.
- [ ] Je connais le rôle du NVIC.
- [ ] Je connais les registres NVIC (ISER, ICER, IPR, ISPR…).
- [ ] Je comprends la préemption et la sous-priorité.
- [ ] Je sais mapper une broche sur une ligne EXTI via AFIO_EXTICR.
- [ ] Je connais les registres EXTI (IMR, RTSR, FTSR, PR).
- [ ] Je sais configurer EXTI en CubeMX.
- [ ] Je sais écrire un `HAL_GPIO_EXTI_Callback`.
- [ ] Je sais acquitter un pending en version registres.
- [ ] Je connais les bonnes pratiques d'une ISR.
- [ ] J'ai implémenté un compteur d'impulsions avec EXTI.
- [ ] J'ai mesuré la latence d'interruption.
- [ ] J'ai rédigé mon compte-rendu de TP4.
- [ ] J'ai lu le chapitre 10 du RM0008.

### 📊 Mon score : ___ / 14

| Score | Interprétation |
|---|---|
| 12–14 | ✅ Prêt pour la S5 (TIMER) |
| 8–11 | ⚠️ Revoir les points manquants |
| < 8 | 🔁 Reprendre les activités 2 à 5 |

---
# 📚 RESSOURCES Semaine 4

### Documents officiels
- 📄 **RM0008** — chapitre 10 (Interrupts and events)
- 📄 **PM0056** (Cortex-M3) — chapitre 4.2 (NVIC)
- 📄 **Datasheet STM32F103x6** — section 2.3.10 (EXTI)

### Outils
- **STM32CubeMX** — onglets *Pinout*, *GPIO* et *NVIC*
- **Oscilloscope** ou **analyseur logique** pour mesurer la latence
- **STM32CubeProgrammer** pour lire les registres EXTI/NVIC en live

### Vidéos
- *STM32 EXTI Tutorial (HAL)* — ControllersTech
- *Cortex-M NVIC Explained* — YouTube (ARM Education)

### Bonnes pratiques
- ISR courte (< quelques µs)
- Variables partagées en `volatile`
- Flag plutôt que traitement lourd dans l'ISR
- Acquitter toujours le pending (HAL le fait, registres à gérer)

---

### 🔗 Passage à la semaine 5

**Prochaine séance :** TIMER — base de temps  
- Structure d'un timer (PSC, ARR, CNT)
- Modes : one-shot, périodique, PWM (aperçu)
- Interruptions de timer
- Calculs de période et de fréquence

**Préparation :** Lire le chapitre 15 (TIM) du RM0008.

---

**Fin du notebook — Semaine 4** ✨